# Judge candidate-bank answers and create class preference pairs

This notebook selects the first 200 unique training questions from each completed strict-extractive candidate bank (0.5B and 3B), labels every generated candidate as C3/C2/C1/UNCERTAIN, and writes question class slates plus DPO-style chosen/rejected pairs.

C3 exact matches are deterministic. C2, C1, and UNCERTAIN are assigned by the biomedical LLM judge using the question, accepted aliases, and supplied snippets. The run is cached and resumable. Review the model-judged records before using the pairs for a final training experiment.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path('/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining')
sys.path.insert(0, str(PROJECT_ROOT))
from cse_dpo.candidate_bank_class_judge import CandidateBankClassJudge, JUDGE_SYSTEM, build_pairs, digest, prepare_records

BANK_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/candidate_banks/strict_extractive_qwen25_05b_3b_all_splits_s20_v2_seed3407'
BANK_FILES = {
    '05b': BANK_ROOT / 'qwen25_05b/train/adapter-best-evidence-mrr/candidate_bank.jsonl',
    '3b': BANK_ROOT / 'qwen25_3b/train/adapter-best-evidence-mrr/candidate_bank.jsonl',
}
BIOASQ_FILE = PROJECT_ROOT / 'data/training13b.json'
API_KEY_FILE = PROJECT_ROOT / 'open_ai_api.txt'
OUTPUT_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/class_judgments/candidate_banks_train_first200_c3_c1_v1'

QUESTION_LIMIT = 200
JUDGE_MODEL = os.environ.get('CANDIDATE_BANK_JUDGE_MODEL', 'gpt-4.1-mini-2025-04-14')
JUDGE_ENDPOINT = os.environ.get('CANDIDATE_BANK_JUDGE_ENDPOINT', 'https://api.openai.com/v1/chat/completions')
MAX_NEW_JUDGE_CALLS = None  # Set an integer for a bounded pilot; None annotates all non-exact candidates.
MAX_RETRIES = 2
TIMEOUT_SECONDS = 120
PAIR_POLICY = 'all_ordered_class_pairs'  # Alternative: semantic_positive_vs_negative
INJECT_GOLD_C3 = True  # Add one deterministic gold C3 anchor for questions without generated C3.

for path in [BIOASQ_FILE, API_KEY_FILE, *BANK_FILES.values()]:
    assert path.exists(), path
print('Output:', OUTPUT_ROOT)
print('Fresh run size:', QUESTION_LIMIT * 20 * len(BANK_FILES), 'candidate records')

In [ ]:
records, selected_question_ids, row_counts = prepare_records(
    BANK_FILES, BIOASQ_FILE, question_limit=QUESTION_LIMIT
)
print('Rows by model:', row_counts)
print('Questions by model:', {model: len(ids) for model, ids in selected_question_ids.items()})
print('Question overlap:', len(set(selected_question_ids['05b']) & set(selected_question_ids['3b'])))
print('Candidate records:', len(records))
print('Rubric SHA256:', digest(JUDGE_SYSTEM))
assert len(records) == QUESTION_LIMIT * 20 * len(BANK_FILES)

In [ ]:
judge = CandidateBankClassJudge(
    records=records,
    output_root=OUTPUT_ROOT,
    api_key_file=API_KEY_FILE,
    judge_model=JUDGE_MODEL,
    judge_endpoint=JUDGE_ENDPOINT,
    max_new_judge_calls=MAX_NEW_JUDGE_CALLS,
    max_retries=MAX_RETRIES,
    timeout_seconds=TIMEOUT_SECONDS,
)
judgments, judgment_summary = judge.run()
print(json.dumps(judgment_summary, indent=2))

In [ ]:
pair_summary = build_pairs(records, judgments, OUTPUT_ROOT, policy=PAIR_POLICY, inject_gold_c3=INJECT_GOLD_C3)
manifest = {
    'status': 'complete' if judgment_summary.get('status') == 'complete' else 'incomplete',
    'question_limit_per_model': QUESTION_LIMIT,
    'question_counts': {model: len(ids) for model, ids in selected_question_ids.items()},
    'candidate_counts': row_counts,
    'bank_files': {model: str(path) for model, path in BANK_FILES.items()},
    'judge_model': JUDGE_MODEL,
    'judge_endpoint_sha256': digest(JUDGE_ENDPOINT),
    'rubric_sha256': digest(JUDGE_SYSTEM),
    'pair_policy': PAIR_POLICY,
    'inject_gold_c3': INJECT_GOLD_C3,
    'judgment_summary': judgment_summary,
    'pair_summary': pair_summary,
}
(OUTPUT_ROOT / 'summary.json').write_text(json.dumps({**judgment_summary, **pair_summary}, indent=2) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print(json.dumps(pair_summary, indent=2))
print('Judgments:', OUTPUT_ROOT / 'candidate_class_judgments.jsonl')
print('Slates:', OUTPUT_ROOT / 'question_class_slates.jsonl')
print('Pairs:', OUTPUT_ROOT / 'dpo_pairs.jsonl')

In [ ]:
from collections import Counter
judgment_rows = [json.loads(line) for line in (OUTPUT_ROOT / 'candidate_class_judgments.jsonl').open(encoding='utf-8') if line.strip()]
pair_rows = [json.loads(line) for line in (OUTPUT_ROOT / 'dpo_pairs.jsonl').open(encoding='utf-8') if line.strip()]
print('Class counts:', dict(Counter(row['class'] for row in judgment_rows)))
print('Pair count:', len(pair_rows))
for row in pair_rows[:5]:
    print(row['question_id'], row['chosen_class'], '>', row['rejected_class'], '|', row['chosen'], 'vs', row['rejected'])